# Random Forest

In [ ]:
import pandas as pd
from joblib import Parallel, delayed
import numpy as np
import math
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer, classification_report, roc_auc_score
import tqdm
from tabulate import tabulate
from pathlib import Path
import warnings
import time
from datetime import timedelta
import re
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

"""
    
'dataset_fuso': FILE_PATH / 'dataset_fuso.csv'
"""


# Lista dei csv su cui fare training
datasets = {
't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
    
}

# Training

In [ ]:
def training(file_path, csv_name):

    df = pd.read_csv(file_path)

    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']
    
    df_validi = df.dropna(subset=original_target_list).copy()
    
    if len(df_validi) < 10:
        print(f"ATTENZIONE: Dataset {csv_name} ha troppi pochi campioni validi ({len(df_validi)}). Skip.")
        return None

    # Binarizzazione Target
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    df_validi['HER2_class'] = (df_validi['HER2 [SII]'] >= 3).astype(int) # Nota: >=3 per HER2

    final_target_list = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']
    
    # Preparazione Features
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')
    
    # Pulizia nomi colonne e imputazione
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]
    features = features.fillna(features.mean())

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    cv = GroupKFold(n_splits=5) # 5 fold su 32 pazienti sono circa 6 pazienti per fold.

    base_model = RandomForestClassifier(random_state=42, n_jobs=1)
    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
        'estimator__n_estimators': [100, 300],        
        'estimator__max_depth': [1, 2, 3],            
        'estimator__min_samples_leaf': [2, 3, 4],    
        'estimator__max_features': ['log2', 'sqrt'], 
        'estimator__class_weight': ['balanced', None] 
    }

    # Calcolo la f1_score per le 4 classi
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            s = f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0)
            scores.append(s)
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    # Grid Search CV
    total_combinations = math.prod(len(v) for v in iperparametri.values())
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name} (N={len(features)})")

    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        refit=True,
        error_score='raise'
    )

    grid_search.fit(features, target, groups=groups)

    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    fold_reports = []
    for train_idx, test_idx in cv.split(features, target, groups):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        model_clone = grid_search.best_estimator_
        model_clone.fit(X_train, y_train)
        
        y_pred = model_clone.predict(X_test)
        
        # Probabilità per AUC per un target
        try:
            y_proba_list = model_clone.predict_proba(X_test)
        except:
            y_proba_list = [None] * len(final_target_list)

        fold_report = {}
        for i, col_name in enumerate(final_target_list):
            # Classification Report
            rep = classification_report(
                y_test.iloc[:, i], 
                y_pred[:, i], 
                output_dict=True, 
                zero_division=0
            )

            # Calcolo AUC sicuro
            auc_val = np.nan
            if y_proba_list[i] is not None:
                # Gestione casi monoclasse nel test set
                unique_classes = np.unique(y_test.iloc[:, i])
                if len(unique_classes) == 2:
                    try:
                        # Prendi probabilità classe 1. 
                        # MultiOutputClassifier restituisce una lista di array (n_samples, n_classes)
                        probs = y_proba_list[i]
                        if probs.shape[1] == 2:
                            auc_val = roc_auc_score(y_test.iloc[:, i], probs[:, 1])
                        else:
                            auc_val = 0.5 # Modello non ha predetto probabilità utili
                    except ValueError:
                        pass
            
            rep['auc'] = auc_val
            fold_report[col_name] = rep
        
        fold_reports.append(fold_report)

    # Costruzione Risultato Finale pulita
    clean_params = {k.replace('estimator__', ''): v for k, v in best_params.items()}
    
    final_result = [{
        **clean_params,
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports
    }]

    return final_result

# Stampo i risultati in un formato piú leggibile

In [ ]:
def print_grid_search_results(results_per_dataset):

    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI (Random Forest)")
    print("=" * 80)

    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        if not metrics_list: continue 
        
        best_result = metrics_list[0]

        # Calcolo AUC Media 
        auc_values = []
        if best_result.get('fold_reports'):
            for fold_rep in best_result['fold_reports']:
                for target_metrics in fold_rep.values():
                    if isinstance(target_metrics, dict) and 'auc' in target_metrics:
                        val = target_metrics['auc']
                        if not np.isnan(val):
                            auc_values.append(val)
        
        mean_auc = np.mean(auc_values) if auc_values else 0.0

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}")
        print(f" Mean AUC    = {mean_auc:.3f}\n") 

        print("Iperparametri Ottimali:")
        
        # Mappatura per la tabella dettagliata
        possible_params = [
            ('N. Estimators', 'n_estimators'),      
            ('Max Depth', 'max_depth'),             
            ('Min Samples Leaf', 'min_samples_leaf'), 
            ('Max Features', 'max_features'),       
            ('Class Weight', 'class_weight')        
        ]

        params_table = []
        for label, key in possible_params:
            val = best_result.get(key, "N/A")
            params_table.append([label, val])

        print(tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))

        print("\n Metriche (Esempio primo fold valido):\n")
        
        if best_result['fold_reports']:
            first_rep = best_result['fold_reports'][0]
            
            for target_name, metrics in first_rep.items():
                # Titolo Target con AUC specifica
                auc_v = metrics.get('auc', np.nan)
                auc_str = f"{auc_v:.3f}" if not np.isnan(auc_v) else "N/A"
                print(f"  Target: {target_name} (AUC: {auc_str})")
                
                # Tabella Classi
                rows = []
                for cls in ['0', '1']:
                    if cls in metrics:
                        d = metrics[cls]
                        rows.append([
                            f"Classe {cls}",
                            f"{d['precision']:.3f}",
                            f"{d['recall']:.3f}",
                            f"{d['f1-score']:.3f}",
                            d['support']
                        ])
                print(tabulate(rows, headers=['', 'Precision', 'Recall', 'F1', 'Supp'], tablefmt='plain'))
                print("-" * 40)

        # Dati per tabellone finale
        summary_data.append([
            name,
            best_result['mean_score'],
            best_result['std_score'],
            mean_auc,
            best_result.get('n_estimators', 0),
            best_result.get('min_samples_leaf', 0),
            best_result.get('max_depth', 0)
        ])

    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO FINALE")
    print("=" * 80 + "\n")

    summary_data.sort(key=lambda x: x[1], reverse=True) 

    print(tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std', 'AUC', 'Est', 'Leaf', 'Depth'],
                   tablefmt='grid',
                   floatfmt=('.3f', '.3f', '.3f', '.3f', '.0f', '.0f', '.0f')))

# Lettura dei file

In [ ]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)




end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")
